In [0]:
# Databricks - Gold data quality checks
# File: databricks/data_quality/01_gold_checks.py

from datetime import datetime, UTC
from pyspark.sql import functions as F

ACCOUNT = "storagedatalake9105"
GOLD_BASE = f"abfss://gold@{ACCOUNT}.dfs.core.windows.net/adventureworks"
DQ_REPORT_PATH = f"{GOLD_BASE}/quality/data_quality_report"
DQ_REPORT_TABLE = "gold.data_quality_report"


def main() -> None:
    fact = spark.table("gold.fact_sales")
    dim_customer = spark.table("gold.dim_customer")
    dim_product = spark.table("gold.dim_product")
    dim_territory = spark.table("gold.dim_territory")
    dim_date = spark.table("gold.dim_date")

    checks = [
        ("missing_customers", fact.join(dim_customer, "CustomerKey", "left_anti").count()),
        ("missing_products", fact.join(dim_product, "ProductKey", "left_anti").count()),
        ("missing_territories", fact.join(dim_territory, "TerritoryKey", "left_anti").count()),
        ("missing_dates", fact.join(dim_date, "date_key", "left_anti").count()),
        ("null_customer", fact.filter(F.col("CustomerKey").isNull()).count()),
        ("null_product", fact.filter(F.col("ProductKey").isNull()).count()),
        ("null_date", fact.filter(F.col("date_key").isNull()).count()),
        ("negative_revenue", fact.filter(F.col("revenue") < 0).count()),
        ("negative_profit", fact.filter(F.col("profit") < 0).count()),
        ("invalid_quantity", fact.filter(F.col("OrderQuantity") <= 0).count()),
    ]

    run_ts = datetime.now(UTC).isoformat()

    report_df = spark.createDataFrame(
        [(name, failed, run_ts) for name, failed in checks],
        ["check_name", "failed_rows", "run_utc"],
    )

    (
        report_df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .save(DQ_REPORT_PATH)
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DQ_REPORT_TABLE}
        USING DELTA
        LOCATION '{DQ_REPORT_PATH}'
    """)

    print("Gold data quality checks complete.")
    report_df.orderBy("check_name").show(truncate=False)


if __name__ == "__main__":
    main()
